### 1. Configurations

In [0]:
!pip install xgboost
%restart_python

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pyspark.sql.functions import col, avg
import numpy as np

import os
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/main/models/model"

CATALOG = "main"
SCHEMA  = "synthetic"


### 2. Load the silver tables

In [0]:
claims    = spark.table(f"{CATALOG}.{SCHEMA}.claims")
providers = spark.table(f"{CATALOG}.{SCHEMA}.providers")
diagnosis = spark.table(f"{CATALOG}.{SCHEMA}.diagnosis")
cost      = spark.table(f"{CATALOG}.{SCHEMA}.cost")

print("claims:", claims.count())
print("providers:", providers.count())
print("diagnosis:", diagnosis.count())
print("cost:", cost.count())

claims: 5000
providers: 21
diagnosis: 6
cost: 12


### 3. Create Gold Table (gold_claim_base)

In [0]:
from pyspark.sql.functions import avg

# Aggregate cost by procedure_code (average across regions)
cost_agg = cost.groupBy("procedure_code").agg(
    avg("average_cost").alias("average_cost"),
    avg("expected_cost").alias("expected_cost")
)

print("cost_agg row count:", cost_agg.count())

# Join the Tables
gold_claim_base = claims.join(providers, on="provider_id", how="left")
gold_claim_base = gold_claim_base.join(diagnosis, on="diagnosis_code", how="left")
gold_claim_base = gold_claim_base.join(cost_agg, on="procedure_code", how="left")

print("gold_claim_base row count:", gold_claim_base.count())

cost_agg row count: 6
gold_claim_base row count: 5000


### 4. handling Null Values

In [0]:
# Check nulls Counts
from pyspark.sql.functions import when, isnull, count as spark_count

null_counts = gold_claim_base.select([
    spark_count(when(isnull(c), c)).alias(c)
    for c in gold_claim_base.columns
])
null_counts.show()

+--------------+--------------+-----------+--------+----------+-------------+----+-----------+-----------+---------+--------+--------+--------+------------+-------------+
|procedure_code|diagnosis_code|provider_id|claim_id|patient_id|billed_amount|date|denial_flag|doctor_name|specialty|location|category|severity|average_cost|expected_cost|
+--------------+--------------+-----------+--------+----------+-------------+----+-----------+-----------+---------+--------+--------+--------+------------+-------------+
|           303|           447|          0|       0|         0|          472|   0|          0|          0|        0|     171|     447|     447|         303|          303|
+--------------+--------------+-----------+--------+----------+-------------+----+-----------+-----------+---------+--------+--------+--------+------------+-------------+



In [0]:
# Replace Null values with Unknows

gold_claim_base = gold_claim_base.fillna({
    "category"      : "Unknown",
    "severity"      : "Unknown",
    "average_cost"  : 0.0,
    "expected_cost" : 0.0,
    "specialty"     : "Unknown",
    "location"      : "Unknown",
    "doctor_name"   : "Unknown"
})

print("Nulls values replaced with Unknown.")

Nulls values replaced with Unknown.


### 5. Save the Gold Table

In [0]:
gold_claim_base.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.gold.gold_claim_base")

print("gold_claim_base saved.")

gold_claim_base saved.


### 6. Feature Engineering

In [0]:
from pyspark.sql.functions import col, when, count, avg, round

# billed vs average cost ratio
# high cost flag
gold_claim_features = gold_claim_base.withColumn(
    "billed_vs_avg_cost", round(col("billed_amount") / (col("average_cost") + 1), 2)
).withColumn(
    "high_cost_flag", when(col("billed_amount") > col("average_cost"), 1).otherwise(0)
)

In [0]:
from pyspark.sql.functions import count as spark_count, sum as spark_sum

# Provider claim count
provider_stats = gold_claim_base.groupBy("provider_id").agg(
    spark_count("claim_id").alias("provider_claim_count"),
    round(spark_sum("denial_flag") / spark_count("claim_id"), 4).alias("provider_risk_score")
)

gold_claim_features = gold_claim_features.join(provider_stats, on="provider_id", how="left")

In [0]:
# Diagnosis count per code
diagnosis_stats = gold_claim_base.groupBy("diagnosis_code").agg(
    spark_count("claim_id").alias("diagnosis_count")
)

# Claim frequency per patient
claim_freq = gold_claim_base.groupBy("patient_id").agg(
    spark_count("claim_id").alias("claim_frequency")
)

gold_claim_features = gold_claim_features.join(diagnosis_stats, on="diagnosis_code", how="left")
gold_claim_features = gold_claim_features.join(claim_freq, on="patient_id", how="left")

In [0]:
# Map severity text to numeric score
gold_claim_features = gold_claim_features.withColumn(
    "severity_score",
    when(col("severity") == "High", 3)
    .when(col("severity") == "Medium", 2)
    .when(col("severity") == "Low", 1)
    .otherwise(0)
)

In [0]:
print("Feature table row count:", gold_claim_features.count())
gold_claim_features.show(5)

gold_claim_features.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{CATALOG}.gold.gold_claim_features")

print("gold_claim_features saved.")

Feature table row count: 5000
+----------+--------------+-----------+--------------+--------+-------------+----------+-----------+-----------+----------+---------+--------+--------+------------+-------------+------------------+--------------+--------------------+-------------------+---------------+---------------+--------------+
|patient_id|diagnosis_code|provider_id|procedure_code|claim_id|billed_amount|      date|denial_flag|doctor_name| specialty| location|category|severity|average_cost|expected_cost|billed_vs_avg_cost|high_cost_flag|provider_claim_count|provider_risk_score|diagnosis_count|claim_frequency|severity_score|
+----------+--------------+-----------+--------------+--------+-------------+----------+-----------+-----------+----------+---------+--------+--------+------------+-------------+------------------+--------------+--------------------+-------------------+---------------+---------------+--------------+
|      P704|          NULL|      PR112|         PROC6|  C00001|    

### 7. Preparing Dataset for ML Model

In [0]:
from pyspark.sql.functions import col

# Select only ML-relevant columns
ml_cols = [
    "billed_amount",
    "billed_vs_avg_cost",
    "high_cost_flag",
    "provider_claim_count",
    "provider_risk_score",
    "diagnosis_count",
    "claim_frequency",
    "severity_score",
    "specialty",
    "category",
    "location",
    "denial_flag"  # target
]

ml_df = gold_claim_features.select(ml_cols)

# Fill remaining nulls
ml_df = ml_df.fillna({
    "diagnosis_count"    : 0,
    "provider_risk_score": 0.0,
    "billed_vs_avg_cost" : 0.0,
    "claim_frequency"    : 0
})

print("Null check after fill:")
from pyspark.sql.functions import when, isnull, count as spark_count
ml_df.select([spark_count(when(isnull(c), c)).alias(c) for c in ml_df.columns]).show()

Null check after fill:
+-------------+------------------+--------------+--------------------+-------------------+---------------+---------------+--------------+---------+--------+--------+-----------+
|billed_amount|billed_vs_avg_cost|high_cost_flag|provider_claim_count|provider_risk_score|diagnosis_count|claim_frequency|severity_score|specialty|category|location|denial_flag|
+-------------+------------------+--------------+--------------------+-------------------+---------------+---------------+--------------+---------+--------+--------+-----------+
|          472|                 0|             0|                   0|                  0|              0|              0|             0|        0|       0|       0|          0|
+-------------+------------------+--------------+--------------------+-------------------+---------------+---------------+--------------+---------+--------+--------+-----------+



In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import dense_rank

# Manually encode categorical columns using dense_rank to avoid large ML models
cat_cols = ["specialty", "category", "location"]

for c in cat_cols:
    window = Window.orderBy(c)
    ml_df = ml_df.withColumn(c + "_idx", dense_rank().over(window) - 1)

# Drop original categorical columns
ml_df = ml_df.drop(*cat_cols)

ml_df.show(3)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------+------------------+--------------+--------------------+-------------------+---------------+---------------+--------------+-----------+-------------+------------+------------+
|billed_amount|billed_vs_avg_cost|high_cost_flag|provider_claim_count|provider_risk_score|diagnosis_count|claim_frequency|severity_score|denial_flag|specialty_idx|category_idx|location_idx|
+-------------+------------------+--------------+--------------------+-------------------+---------------+---------------+--------------+-----------+-------------+------------+------------+
|      4180.14|              1.02|             1|                 174|             0.2299|            740|              4|             3|          0|            0|           0|           0|
|         NULL|               0.0|             0|                 174|             0.2299|            740|              5|             3|          1|            0|           0|           0|
|     11430.97|              1.87|             1| 

In [0]:
from pyspark.ml.feature import VectorAssembler

# Use the already-encoded ml_df from Cell 21
# Define feature columns for VectorAssembler
feature_cols = [
    "billed_amount",
    "billed_vs_avg_cost",
    "high_cost_flag",
    "provider_claim_count",
    "provider_risk_score",
    "diagnosis_count",
    "claim_frequency",
    "severity_score",
    "specialty_idx",
    "category_idx",
    "location_idx"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
ml_df_vectorized = assembler.transform(ml_df)

# Keep only features + target
final_df = ml_df_vectorized.select("features", "denial_flag")

# Split
train_df, test_df = final_df.randomSplit([0.7, 0.3], seed=42)

print("Train size:", train_df.count())
print("Test size:", test_df.count())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Train size: 3237
Test size: 1291


### 8. Logistic Regression Model using MLFlow

In [0]:
# Checking Logistic Regression Model Accuracy

import mlflow
import mlflow.spark
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

mlflow.set_experiment("/Users/deepeshvcd6273@gmail.com/claim_denial_prediction")

with mlflow.start_run(run_name="logistic_regression"):

    lr = LogisticRegression(featuresCol="features", labelCol="denial_flag", maxIter=100)
    lr_model = lr.fit(train_df)

    predictions = lr_model.transform(test_df)

    # Evaluate
    binary_eval  = BinaryClassificationEvaluator(labelCol="denial_flag")
    multi_eval   = MulticlassClassificationEvaluator(labelCol="denial_flag", metricName="accuracy")
    precision_eval = MulticlassClassificationEvaluator(labelCol="denial_flag", metricName="weightedPrecision")
    recall_eval    = MulticlassClassificationEvaluator(labelCol="denial_flag", metricName="weightedRecall")

    roc_auc   = binary_eval.evaluate(predictions)
    accuracy  = multi_eval.evaluate(predictions)
    precision = precision_eval.evaluate(predictions)
    recall    = recall_eval.evaluate(predictions)

    # Log params + metrics
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("maxIter", 100)
    mlflow.log_metric("roc_auc",   roc_auc)
    mlflow.log_metric("accuracy",  accuracy)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall",    recall)

    # Log model
    mlflow.spark.log_model(lr_model, "lr_model")

    print(f"Logistic Regression Results:")
    print(f"  Accuracy  : {accuracy:.4f}")
    print(f"  Precision : {precision:.4f}")
    print(f"  Recall    : {recall:.4f}")
    print(f"  ROC-AUC   : {roc_auc:.4f}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
2026/05/04 11:23:47 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.0.5) contains a local version label (+databricks.connect.18.0.5). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/05/04 11:23:50 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-1cc9e583-58e6-4371-99f8-6d/tmpqxfzf28l/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full

Logistic Regression Results:
  Accuracy  : 0.8699
  Precision : 0.8694
  Recall    : 0.8699
  ROC-AUC   : 0.9279


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score

# Retrain without leakage features
feature_cols_clean = [
    "billed_amount",
    "billed_vs_avg_cost",
    "high_cost_flag",
    "severity_score",
    "specialty_idx",
    "category_idx",
    "location_idx"
]

assembler_clean = VectorAssembler(
    inputCols=feature_cols_clean, 
    outputCol="features", 
    handleInvalid="skip"
)
ml_df_clean = assembler_clean.transform(ml_df_vectorized.drop("features"))

final_df_clean = ml_df_clean.select("features", "denial_flag")
train_clean, test_clean = final_df_clean.randomSplit([0.7, 0.3], seed=42)

# Convert to pandas
train_pd_clean = train_clean.toPandas()
test_pd_clean  = test_clean.toPandas()

X_train_clean = np.vstack(train_pd_clean["features"].apply(lambda v: np.array(v.toArray())))
y_train_clean = train_pd_clean["denial_flag"].values
X_test_clean  = np.vstack(test_pd_clean["features"].apply(lambda v: np.array(v.toArray())))
y_test_clean  = test_pd_clean["denial_flag"].values

# Retrain
with mlflow.start_run(run_name="xgboost_no_leakage"):
    xgb_clean = xgb.XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        eval_metric="logloss"
    )
    xgb_clean.fit(X_train_clean, y_train_clean)

    y_pred_clean       = xgb_clean.predict(X_test_clean)
    y_pred_proba_clean = xgb_clean.predict_proba(X_test_clean)[:, 1]

    roc_auc_clean   = roc_auc_score(y_test_clean, y_pred_proba_clean)
    accuracy_clean  = accuracy_score(y_test_clean, y_pred_clean)
    precision_clean = precision_score(y_test_clean, y_pred_clean, average="weighted")
    recall_clean    = recall_score(y_test_clean, y_pred_clean, average="weighted")

    mlflow.log_param("model", "XGBoost_no_leakage")
    mlflow.log_metric("roc_auc",   roc_auc_clean)
    mlflow.log_metric("accuracy",  accuracy_clean)
    mlflow.log_metric("precision", precision_clean)
    mlflow.log_metric("recall",    recall_clean)

    print(f"XGBoost (No Leakage) Results:")
    print(f"  Accuracy  : {accuracy_clean:.4f}")
    print(f"  Precision : {precision_clean:.4f}")
    print(f"  Recall    : {recall_clean:.4f}")
    print(f"  ROC-AUC   : {roc_auc_clean:.4f}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


XGBoost (No Leakage) Results:
  Accuracy  : 0.8598
  Precision : 0.8589
  Recall    : 0.8598
  ROC-AUC   : 0.9422


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


### 9. Register Model to MLFlow

In [0]:
from mlflow.models.signature import infer_signature

# Build signature from clean model
input_example = X_test_clean[:3]
output_example = xgb_clean.predict(input_example)
signature = infer_signature(input_example, output_example)

with mlflow.start_run(run_name="xgboost_final") as run:
    mlflow.log_param("model", "XGBoost_no_leakage")
    mlflow.log_metric("roc_auc",   roc_auc_clean)
    mlflow.log_metric("accuracy",  accuracy_clean)
    mlflow.log_metric("precision", precision_clean)
    mlflow.log_metric("recall",    recall_clean)

    mlflow.xgboost.log_model(
        xgb_clean,
        "xgb_model",
        signature=signature,
        input_example=input_example
    )

    run_id = run.info.run_id
    print(f"Run ID: {run_id}")

# Register to Unity Catalog
model_uri = f"runs:/{run_id}/xgb_model"
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name=f"{CATALOG}.{SCHEMA}.claim_denial_model"
)

print(f"Model registered: {registered_model.name}")
print(f"Version: {registered_model.version}")

2026/05/04 11:26:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-2eeeec2a-2484.cloud.databricks.com/ml/experiments/2477609430828014/models/m-8775bddc01794ad384e9984d9901e944?o=7474644751098620


Run ID: 6fcf9b66fc8a4d68b5ff8a7e22a4e0f1


Registered model 'main.synthetic.claim_denial_model' already exists. Creating a new version of this model...
2026/05/04 11:26:07 WARNING mlflow.tracking._model_registry.fluent: Run with id 6fcf9b66fc8a4d68b5ff8a7e22a4e0f1 has no artifacts at artifact path 'xgb_model', registering model based on models:/m-8775bddc01794ad384e9984d9901e944 instead


Uploading artifacts:   0%|          | 0/12 [00:00<?, ?it/s]

🔗 Created version '2' of model 'main.synthetic.claim_denial_model': https://dbc-2eeeec2a-2484.cloud.databricks.com/explore/data/models/main/synthetic/claim_denial_model/version/2?o=7474644751098620


Model registered: main.synthetic.claim_denial_model
Version: 2


### Sample Prediction

In [0]:
test_claims = np.array([
    [15000.0, 2.5,  1, 3, 0, 4, 3],   # High risk — Cardiology, Heart, Hyderabad
    [20000.0, 3.8,  1, 3, 0, 2, 4],   # High risk — Cardiology, Diabetes, Mumbai
    [25000.0, 4.5,  1, 3, 0, 6, 5],   # High risk — Cardiology, Unknown, Unknown
    [3000.0,  0.8,  0, 1, 2, 5, 0],   # Low risk  — Neurology, Skin, Bangalore
    [1500.0,  0.5,  0, 1, 1, 1, 1],   # Low risk  — General, Cold, Chennai
    [4500.0,  1.0,  0, 1, 2, 3, 0],   # Low risk  — Neurology, Fever, Bangalore
    [7000.0,  1.4,  1, 2, 0, 2, 4],   # Medium    — Cardiology, Diabetes, Mumbai
    [5000.0,  1.0,  0, 1, 1, 3, 2],   # Normal    — General, Fever, Delhi
    [4200.0,  0.95, 0, 1, 2, 5, 0],   # Normal    — Neurology, Skin, Bangalore
    [3800.0,  0.9,  0, 1, 1, 1, 1],   # Normal    — General, Cold, Chennai
])

claim_labels = [
    "High risk  — Cardiology, Heart, Hyderabad",
    "High risk  — Cardiology, Diabetes, Mumbai",
    "High risk  — Cardiology, Unknown, Unknown",
    "Low risk   — Neurology, Skin, Bangalore",
    "Low risk   — General, Cold, Chennai",
    "Low risk   — Neurology, Fever, Bangalore",
    "Medium     — Cardiology, Diabetes, Mumbai",
    "Normal     — General, Fever, Delhi",
    "Normal     — Neurology, Skin, Bangalore",
    "Normal     — General, Cold, Chennai",
]

probs       = xgb_clean.predict_proba(test_claims)[:, 1]
predictions = xgb_clean.predict(test_claims)

print(f"{'#':<3} {'Description':<45} {'Prob':>6}  {'Prediction':<10} {'Risk'}")
print("-" * 85)
for i, (label, prob, pred) in enumerate(zip(claim_labels, probs, predictions)):
    risk    = "HIGH" if prob >= 0.5 else "LOW"
    outcome = "DENIED" if pred == 1 else "APPROVED"
    print(f"{i+1:<3} {label:<45} {prob:.4f}  {outcome:<10} {risk}")

#   Description                                     Prob  Prediction Risk
-------------------------------------------------------------------------------------
1   High risk  — Cardiology, Heart, Hyderabad     0.9990  DENIED     HIGH
2   High risk  — Cardiology, Diabetes, Mumbai     0.9990  DENIED     HIGH
3   High risk  — Cardiology, Unknown, Unknown     0.9988  DENIED     HIGH
4   Low risk   — Neurology, Skin, Bangalore       0.0009  APPROVED   LOW
5   Low risk   — General, Cold, Chennai           0.0015  APPROVED   LOW
6   Low risk   — Neurology, Fever, Bangalore      0.0015  APPROVED   LOW
7   Medium     — Cardiology, Diabetes, Mumbai     0.8183  DENIED     HIGH
8   Normal     — General, Fever, Delhi            0.0028  APPROVED   LOW
9   Normal     — Neurology, Skin, Bangalore       0.0009  APPROVED   LOW
10  Normal     — General, Cold, Chennai           0.0015  APPROVED   LOW
